# E2SFCA Analysis for EmOC Accessibility
## Kano State, Nigeria

This notebook implements the Enhanced Two-Step Floating Catchment Area (E2SFCA) method to measure spatial accessibility to Emergency Obstetric Care (EmOC) facilities.

**Prerequisites**: Complete Notebook 01 (Data Preparation) first

This notebook:
1. Loads processed data from Notebook 01
2. Calculates distance matrices between demand and supply
3. Applies E2SFCA method for each facility category
4. Calculates accessibility scores
5. Classifies areas by deprivation level
6. Saves results for analysis in Notebook 03

**Method**: E2SFCA (Luo & Qi, 2009)[]

> Note: This notebook requires the [environment dependencies](requirements.txt) to be installed
> as well as either an [openrouteservice API key](https://openrouteservice.org/dev/#/signup) or a local instance of the ORS server.

## Workflow Summary:

The notebook gives an overview of the distribution of centres offering EmOC in the city, their classification and how they can be accessed during an emergency. Open source data from OpenStreetMap and tools (such as the openrouteservice) were used to create accessibility measures. Spatial analysis and other data analytics functions led to generating outputs within the 100x100m grid cells that categorised them into three levels: low, medium, and high.

* **Preprocessing**: Get data for EmOC facilities.
* **Analysis for Offer**:
    * Filter or classify EmOC facilities based on discussed criteria.
    * Visualise EmOC faccilities in their categories.
* **Analysis for Accessibility**:
    * Compute travel times to facilities using openrouteservice API or other routing services.
    * Generate areas for low, medium and high categories based on discussed criteria.
* **Analysis for Demmand**:
    * Downscale the popluation data to the 100x100m grid cells.
    * Derive socio-economic descriptors based on discussed criteria.

* **Result**: Generate results as GIS-compatible files.


### Datasets and Tools:
* [openrouteservice](https://openrouteservice.org/) - generate OD-Matrix on the OpenStreetMap road network

#  Workflow

Make sure you have the required packages installed. You can install them using pip:

```bash
pip install -r requirements.txt
```

This study integrates various Python geospatial analysis libraries and packages to support spatial data processing, visualization, and isochrone generation. The os module is used to interact with the operating system, managing file paths and reading environment variables such as API keys. folium library along with its MarkerCluster plugin, facilitates the creation of interactive maps for visualizing large-scale geospatial data. The openrouteservice.client serves as an interface to the OpenRouteService API, enabling the extraction of isochrones. pandas library for data analysis, provides functions for analyzing, cleaning, exploring, and manipulating data, while fiona supports reading and writing real-world data using multi-layered GIS formats, such as shapefiles. The shapely package is employed for the manipulation and analysis of planar geometric objects.

## Setting up the virtual environment

```bash
# Create a new virtual environment
python -m venv .venv
activate .venv/bin/activate
pip install -r requirements.txt
```

## To run your notebook in VS Code

```bash
pip install -U ipykernel
python -m ipykernel install --user --name=.venv
```

In [1]:
import geopandas as gpd
import os
import numpy as np
import pandas as pd

import openrouteservice
from dotenv import load_dotenv

import rasterio
from rasterio.mask import mask

from shapely.geometry import Point
from pathlib import Path
from shapely.geometry import Polygon
from shapely.geometry import box

import requests
import math
from math import *
from sklearn.preprocessing import MinMaxScaler

### Setting up relevant processing folders

There are different data sources used across the notebook. To handle these data sets, it is recommended to use three directories for input, temp and output data. Some of the files are related to healthcare facilities, population data. The healthcare facilities data is usualy the result of gathering global or national datasets and then carrying out local validation according to the local context. 

Despite being official, administrative boundaries may not reflect the actual patterns of human settlement or economic activity. Therefore, the team used the Functional Urban Area (FUA) as a complementary definition of the study areas. The FUA is defined by [the Joint Research Centre of the European Commission](https://commission.europa.eu/about/departments-and-executive-agencies/joint-research-centre_en) as the actual urban sprawl and human activities, encompassing the core city and economically or socially integrated surrounding regions. The FUA was obtained from [the Global Human Settlement Layer (GHSL) ](https://human-settlement.emergency.copernicus.eu/)dataset, which provides spatial data for functional urban areas worldwide. 

The following datasets are considered as input data for the analysis:


* [Datasets of health facilities](../scripts/Kano/data-inputs/healthcare_facilities.geojson)
* [Population: Women in childbearing age](../scripts/Kano/data-inputs/kano_nga_f_15_49_2015_1km.tif) from [WorldPop](https://hub.worldpop.org/geodata/summary?id=18447)
* [Study Area](../../../docs/study-areas/grid-boundary-kano.gpkg) defined by the IDEAMAPS team

In [2]:
# Set paths to access Kano data
# Define directories
data_inputs = '../Kano/Data/raw/'
data_temp = '../Kano/Data/processed/'
model_outputs = '../Kano/Data/outputs/'

## Spatial Analysis Pipeline

### Travel time and dista calculation using OpenRouteService (ORS)

Using OpenRouteService (ORS) Matrix API to calculate the travel time and distance from each population grid centroid to the healthcare facility. There are two options to process the time and distance calculations: Using the public ORS API or using a local instance of the ORS server.

note: this will generate a file 'OD_matrix_healthcare_pop_grid‘

### Setting up the public API Key from OpenRouteService
In this study, users must obtain an ORS Matrix API key from the [OpenRouteService](https://openrouteservice.org/) platform and subsequently interacted with the OpenRouteService API through the instantiation of the OpenRouteService client. This is the OpenRouteService [API documentation](https://openrouteservice.org/dev/#/api-docs/introduction) for ORS Core-Version 9.0.0. 

Generate a [API Key](https://openrouteservice.org/dev/#/home?tab=1) (Token) it is necessary to sign up at the OpenRouteService dashboard by using your E-mail address or sign up with your GitHub. After logging in, go to the Dashboard by clicking on your profile icon and navigate to the API Keys section. Click "Create API Key" to generate a free key and then choose a service plan (the free plan has limited requests per day). Copy the API Key and store it securely. 

OpenRouteService primarily uses API keys for authentication. However, if a token is required for certain endpoints, you can send a request with your API key in the Authorization header. This process facilitated various geospatial analysis functions, including isochrone generation.


## Enhanced Two-Step Floating Catchment Area (E2SFCA) method

In [ ]:
# Function
# Gaussian decay: weight = exp(t² / beta), where beta = -d² / ln(W)
# - d: catchment threshold (travel time in seconds)
# - W: weight at the catchment boundary (i.e., when t = d, weight = W)
# - This ensures smooth decay from 1.0 (at t=0) to W=0.01 (at t=d),
#   meaning facilities beyond d still contribute but at <1% weight

from math import *
d = 10 * 60 # try max duration 5/10mins/15mins/20 car, under estimation of travel time and traffic condition realted to the selected data sourse 
W = 0.01
beta = - d ** 2 / log(W)
print(beta)

78173.00674258533


In [3]:
origin_dest = gpd.read_file('distances_duration_3_closest_Emoc.gpkg')
print(origin_dest.head())

   grid_id  origin_lon  origin_lat  origin_lon_min  origin_lat_min  \
0        1    8.301005   12.122137        8.300491       12.121729   
1        1    8.301005   12.122137        8.300491       12.121729   
2        1    8.301005   12.122137        8.300491       12.121729   
3        2    8.319272   12.072376        8.318758       12.071968   
4        2    8.319272   12.072376        8.318758       12.071968   

   origin_lon_max  origin_lat_max  population  bcount  pop_grid_bcount  \
0        8.301519       12.122545   10.921890    10.0            110.0   
1        8.301519       12.122545   10.921890    10.0            110.0   
2        8.301519       12.122545   10.921890    10.0            110.0   
3        8.319786       12.072784   11.756603     1.0              8.0   
4        8.319786       12.072784   11.756603     1.0              8.0   

   pop_grid_pop  duration_seconds  distance_km  hcf_id  \
0    120.140793            374.30         5.77      14   
1    120.140793   

In [4]:
# 1. Convert 'duration' to numeric, coercing errors to NaN
origin_dest = origin_dest.copy()
origin_dest['duration_seconds'] = pd.to_numeric(origin_dest['duration_seconds'], errors='coerce')

In [5]:
# 2. Drop rows with NaN values in 'duration' column
origin_dest = origin_dest.dropna(subset=['duration_seconds'])
origin_dest['grid_id'] = pd.to_numeric(origin_dest['grid_id'], errors='coerce')
origin_dest_acc = origin_dest  # Backup

In [6]:
# 3. Apply Gaussian decay function to calculate the weight of each grid to healthcare 
# facilities based on the travel duration. d is the travel time and beta is the decay 
# parameter previously calculated.
# The weight decreases as the duration increases, meaning facilities that are further away have less impact.
origin_dest_acc['Weight'] = origin_dest_acc['duration_seconds'].apply(lambda d: round(math.exp(-d**2/beta), 8))

In [7]:
# Compute the Weighted Population (Pop_W), the population of each grid cell is multiplied 
# by the corresponding weight to calculate the weighted population.
origin_dest_acc['Pop_W'] = origin_dest_acc['population'] * origin_dest_acc['Weight']
print(len(origin_dest_acc))
origin_dest_acc.head()

1672390


,grid_id,origin_lon,origin_lat,origin_lon_min,origin_lat_min,origin_lon_max,origin_lat_max,population,bcount,pop_grid_bcount,...,duration_seconds,distance_km,hcf_id,facility_name,dest_lon,dest_lat,Local_Validation,geometry,Weight,Pop_W
0,1,8.301005,12.122137,8.300491,12.121729,8.301519,12.122545,10.921890,10.0,110.0,...,374.30,5.77,14,Dawakin Tofa General Hospital,8.331265,12.107341,Public Comprehensive EmOC,"POLYGON ((8.30051 12.12254, 8.30152 12.12254, ...",0.166596,1.819541
1,1,8.301005,12.122137,8.300491,12.121729,8.301519,12.122545,10.921890,10.0,110.0,...,1679.81,27.83,3,Mariya Sanusi General Hospital,8.473443,12.056152,Public Comprehensive EmOC,"POLYGON ((8.30051 12.12254, 8.30152 12.12254, ...",0.000000,0.000000
2,1,8.301005,12.122137,8.300491,12.121729,8.301519,12.122545,10.921890,10.0,110.0,...,1708.62,28.22,145,Waziri Shehu Gidado General Hospital,8.471150,12.058087,Public Comprehensive EmOC,"POLYGON ((8.30051 12.12254, 8.30152 12.12254, ...",0.000000,0.000000
3,2,8.319272,12.072376,8.318758,12.071968,8.319786,12.072784,11.756603,1.0,8.0,...,470.08,5.20,14,Dawakin Tofa General Hospital,8.331265,12.107341,Public Comprehensive EmOC,"POLYGON ((8.31877 12.07278, 8.31979 12.07278, ...",0.059205,0.696052
4,2,8.319272,12.072376,8.318758,12.071968,8.319786,12.072784,11.756603,1.0,8.0,...,1560.17,25.35,3,Mariya Sanusi General Hospital,8.473443,12.056152,Public Comprehensive EmOC,"POLYGON ((8.31877 12.07278, 8.31979 12.07278, ...",0.000000,0.000000


In [8]:
# 4. Sum the weighted population for each grid cell to get the total weighted population (Pop_W) for each grid cell.
origin_dest_sum = origin_dest_acc.groupby(by='hcf_id')['Pop_W'].sum().reset_index()
origin_dest_sum

,hcf_id,Pop_W
0,1,15603.539814
1,2,36512.813787
2,3,10659.897476
3,4,16032.091170
4,5,55939.302647
...,...,...
129,141,1666.569144
130,142,2708.502060
131,143,5616.732398
132,144,23939.736976


In [9]:
# 5. Merge the total weighted population back to the original DataFrame to have a complete dataset for analysis
origin_dest_acc = origin_dest_acc.merge(origin_dest_sum, on='hcf_id')
origin_dest_acc.head()

,grid_id,origin_lon,origin_lat,origin_lon_min,origin_lat_min,origin_lon_max,origin_lat_max,population,bcount,pop_grid_bcount,...,distance_km,hcf_id,facility_name,dest_lon,dest_lat,Local_Validation,geometry,Weight,Pop_W_x,Pop_W_y
0,1,8.301005,12.122137,8.300491,12.121729,8.301519,12.122545,10.921890,10.0,110.0,...,5.77,14,Dawakin Tofa General Hospital,8.331265,12.107341,Public Comprehensive EmOC,"POLYGON ((8.30051 12.12254, 8.30152 12.12254, ...",0.166596,1.819541,4406.686022
1,1,8.301005,12.122137,8.300491,12.121729,8.301519,12.122545,10.921890,10.0,110.0,...,27.83,3,Mariya Sanusi General Hospital,8.473443,12.056152,Public Comprehensive EmOC,"POLYGON ((8.30051 12.12254, 8.30152 12.12254, ...",0.000000,0.000000,10659.897476
2,1,8.301005,12.122137,8.300491,12.121729,8.301519,12.122545,10.921890,10.0,110.0,...,28.22,145,Waziri Shehu Gidado General Hospital,8.471150,12.058087,Public Comprehensive EmOC,"POLYGON ((8.30051 12.12254, 8.30152 12.12254, ...",0.000000,0.000000,8555.357581
3,2,8.319272,12.072376,8.318758,12.071968,8.319786,12.072784,11.756603,1.0,8.0,...,5.20,14,Dawakin Tofa General Hospital,8.331265,12.107341,Public Comprehensive EmOC,"POLYGON ((8.31877 12.07278, 8.31979 12.07278, ...",0.059205,0.696052,4406.686022
4,2,8.319272,12.072376,8.318758,12.071968,8.319786,12.072784,11.756603,1.0,8.0,...,25.35,3,Mariya Sanusi General Hospital,8.473443,12.056152,Public Comprehensive EmOC,"POLYGON ((8.31877 12.07278, 8.31979 12.07278, ...",0.000000,0.000000,10659.897476


In [10]:
# supply value is set to 1 for simplicity (capacity of HCF)
# supply = 1
# in the future, we will link supply with ownership and EmOC service level
origin_dest_acc = origin_dest_acc.rename(columns={'Pop_W_y': 'Pop_W_S'})  # Pop_W_S: Population Weight Sum

In [11]:
# The supply value is set based on the type of healthcare facility, with different weights assigned to public and private facilities.
supply_map = {
    'Public Comprehensive EmOC': 1,
    'Private Comprehensive EmOC': 0.7,
    'Public Basic EmOC': 0.5,
    'Private Basic EmOC': 0.35
}

### Sensitivity Analysis for Supply Map

In [16]:
# ============================================================
# SENSITIVITY ANALYSIS: Testing robustness of supply weights
# ============================================================
# Re-runs E2SFCA with different supply weight scenarios to check
# whether the final accessibility classification is sensitive to
# the choice of weights.

scenarios = {
    'Baseline':      {'Public Comprehensive EmOC': 1.0, 'Private Comprehensive EmOC': 0.7,
                      'Public Basic EmOC': 0.5,  'Private Basic EmOC': 0.35},
    'Equal':         {'Public Comprehensive EmOC': 1.0, 'Private Comprehensive EmOC': 1.0,
                      'Public Basic EmOC': 1.0,  'Private Basic EmOC': 1.0},
    'Public Only':   {'Public Comprehensive EmOC': 1.0, 'Private Comprehensive EmOC': 0.3,
                      'Public Basic EmOC': 0.5,  'Private Basic EmOC': 0.15},
    'Wider Gap':     {'Public Comprehensive EmOC': 1.0, 'Private Comprehensive EmOC': 0.5,
                      'Public Basic EmOC': 0.3,  'Private Basic EmOC': 0.15},
}

def run_e2sfca_with_supply(origin_dest_input, supply_map, d, beta):
    """Re-run E2SFCA steps with a given supply_map."""
    df = origin_dest_input.copy()
    
    df['Weight'] = np.exp(df['duration_seconds']**2 / beta) * W
    df['Pop_W'] = df['population'] * df['Weight']
    
    pop_w_sum = df.groupby('hcf_id')['Pop_W'].sum().reset_index()
    pop_w_sum.columns = ['hcf_id', 'Pop_W_S']
    df = df.merge(pop_w_sum, on='hcf_id', how='left')
    
    df['supply'] = df['Local_Validation'].map(supply_map)
    df['supply_demand_ratio'] = df['supply'] / df['Pop_W_S']
    df['supply_W'] = df['supply_demand_ratio'] * df['Weight']
    df['Accessibility'] = df.groupby('grid_id')['supply_W'].transform('sum')
    df['Accessibility_standard'] = (df['Accessibility'] - df['Accessibility'].min()) / \
                                    (df['Accessibility'].max() - df['Accessibility'].min())
    
    result = df.drop_duplicates(subset='grid_id')[
        ['grid_id', 'Accessibility', 'Accessibility_standard']
    ].copy()
    return result


# Run all scenarios
results = {}
for name, smap in scenarios.items():
    results[name] = run_e2sfca_with_supply(origin_dest, smap, d, beta)
    print(f"✓ {name} complete")

# ---- Results ----
print("\n" + "=" * 60)
print("SENSITIVITY ANALYSIS RESULTS")
print("=" * 60)

# 1. Raw accessibility values (before normalisation)
print("\nRaw Accessibility (before normalisation):")
print(f"  {'Scenario':15s}  {'Mean':>12s}  {'Median':>12s}  {'Std':>12s}")
for name, res in results.items():
    a = res['Accessibility']
    print(f"  {name:15s}  {a.mean():12.6f}  {a.median():12.6f}  {a.std():12.6f}")

# 2. Percentage change in raw values vs Baseline
baseline_raw_mean = results['Baseline']['Accessibility'].mean()
print("\nChange in raw mean vs Baseline:")
for name, res in results.items():
    if name == 'Baseline':
        continue
    change = (res['Accessibility'].mean() - baseline_raw_mean) / baseline_raw_mean * 100
    print(f"  {name:15s}: {change:+.1f}%")

# 3. Spearman correlation (normalised)
baseline = results['Baseline'].set_index('grid_id')['Accessibility_standard']
print("\nSpearman correlation with Baseline (normalised):")
for name, res in results.items():
    if name == 'Baseline':
        continue
    other = res.set_index('grid_id')['Accessibility_standard']
    common = baseline.index.intersection(other.index)
    corr, pval = spearmanr(baseline[common], other[common])
    print(f"  {name:15s}: r = {corr:.4f}  (p = {pval:.2e})")

# 4. Classification agreement
x_threshold = 0.005
y_threshold = 0.020

print(f"\nClassification agreement (x={x_threshold}, y={y_threshold}):")
baseline_class = np.where(baseline >= y_threshold, 0, np.where(baseline >= x_threshold, 1, 2))

for name, res in results.items():
    if name == 'Baseline':
        continue
    other = res.set_index('grid_id')['Accessibility_standard']
    common = baseline.index.intersection(other.index)
    other_class = np.where(other[common] >= y_threshold, 0, 
                  np.where(other[common] >= x_threshold, 1, 2))
    agreement = np.mean(baseline_class[baseline.index.isin(common)] == other_class) * 100
    print(f"  {name:15s}: {agreement:.1f}% grids same category")

✓ Baseline complete
✓ Equal complete
✓ Public Only complete
✓ Wider Gap complete

SENSITIVITY ANALYSIS RESULTS

Raw Accessibility (before normalisation):
  Scenario                 Mean        Median           Std
  Baseline            14.639457      0.000000   1491.682804
  Equal               23.979411      0.000000   2813.188460
  Public Only          8.024097      0.000000    976.593473
  Wider Gap           10.456089      0.000000   1046.974961

Change in raw mean vs Baseline:
  Equal          : +63.8%
  Public Only    : -45.2%
  Wider Gap      : -28.6%

Spearman correlation with Baseline (normalised):
  Equal          : r = 1.0000  (p = 0.00e+00)
  Public Only    : r = 0.9997  (p = 0.00e+00)
  Wider Gap      : r = 1.0000  (p = 0.00e+00)

Classification agreement (x=0.005, y=0.02):
  Equal          : 100.0% grids same category
  Public Only    : 100.0% grids same category
  Wider Gap      : 100.0% grids same category


In [17]:
# Sensitivity analysis on catchment time
catchment_scenarios = {
    '10min': 10 * 60,
    '15min': 15 * 60,
    '20min': 20 * 60,
    '30min': 30 * 60,
}

for name, d_val in catchment_scenarios.items():
    beta_val = -d_val ** 2
    res = run_e2sfca_with_supply(origin_dest, scenarios['Baseline'], d_val, beta_val)
    median_val = res['Accessibility'].median()
    non_tiny = (res['Accessibility_standard'] > 0.001).sum()
    print(f"{name}: median={median_val:.8f}, grids with meaningful access={non_tiny}/{len(res)}")

10min: median=0.00004194, grids with meaningful access=130252/167239
15min: median=0.00006169, grids with meaningful access=156974/167239
20min: median=0.00007106, grids with meaningful access=164934/167239
30min: median=0.00007717, grids with meaningful access=163671/167239


In [18]:
baseline_10 = run_e2sfca_with_supply(origin_dest, scenarios['Baseline'], 10*60, -(10*60)**2)
x_threshold = 0.005
y_threshold = 0.020

base = baseline_10.set_index('grid_id')['Accessibility_standard']
base_class = np.where(base >= y_threshold, 0, np.where(base >= x_threshold, 1, 2))

print("Classification agreement vs 10min baseline:")
for name, d_val in {'15min': 15*60, '20min': 20*60, '30min': 30*60}.items():
    res = run_e2sfca_with_supply(origin_dest, scenarios['Baseline'], d_val, -d_val**2)
    other = res.set_index('grid_id')['Accessibility_standard']
    common = base.index.intersection(other.index)
    other_class = np.where(other[common] >= y_threshold, 0,
                  np.where(other[common] >= x_threshold, 1, 2))
    agreement = np.mean(base_class[base.index.isin(common)] == other_class) * 100
    corr, _ = spearmanr(base[common], other[common])
    print(f"  {name}: {agreement:.1f}% same category, Spearman r={corr:.4f}")

Classification agreement vs 10min baseline:
  15min: 81.5% same category, Spearman r=0.9576
  20min: 71.6% same category, Spearman r=0.8864
  30min: 70.7% same category, Spearman r=0.7671


In [59]:
# Calculate the supply-demand ratio by dividing the supply value by the total weighted population (Pop_W_S) for each grid cell. This ratio provides an indication of the accessibility of healthcare facilities relative to the demand from the population in each grid cell.
origin_dest_acc['supply'] = origin_dest_acc['Local_Validation'].map(supply_map)
origin_dest_acc['supply_demand_ratio'] = origin_dest_acc['supply'] / origin_dest_acc['Pop_W_S']
origin_dest_acc['supply_demand_ratio'] = (
    origin_dest_acc['supply_demand_ratio']
    .replace([np.inf, -np.inf], 0)
    .fillna(0)
)

In [60]:
# Calculate Rj * Weight for Each Grid Cell
origin_dest_acc['supply_W'] = origin_dest_acc['supply_demand_ratio'] * origin_dest_acc.Weight

In [61]:
# Compute Accessibility Index (Ai) for Each Grid Cell
origin_dest_acc['Accessibility'] = origin_dest_acc.groupby('grid_id')['supply_W'].transform('sum')

In [62]:
# Normalize the Accessibility Index using Min-Max Scaling to bring the values between 0 and 1, making it easier to interpret and compare across different grid cells.
scaler = MinMaxScaler()
origin_dest_acc['Accessibility_standard'] = scaler.fit_transform(origin_dest_acc[['Accessibility']])
origin_dest_acc.head()

,grid_id,origin_lon,origin_lat,origin_lon_min,origin_lat_min,origin_lon_max,origin_lat_max,population,bcount,pop_grid_bcount,...,dest_lat,Local_Validation,Weight,Pop_W_x,Pop_W_S,supply,supply_demand_ratio,supply_W,Accessibility,Accessibility_standard
0,1,8.301005,12.122137,8.300491,12.121729,8.301519,12.122545,10.921890,10.0,110.0,...,12.107341,Public Comprehensive EmOC,0.166596,1.819541,4406.686022,1.0,0.000227,0.000038,0.000038,0.004124
1,1,8.301005,12.122137,8.300491,12.121729,8.301519,12.122545,10.921890,10.0,110.0,...,12.056152,Public Comprehensive EmOC,0.000000,0.000000,10659.897476,1.0,0.000094,0.000000,0.000038,0.004124
2,1,8.301005,12.122137,8.300491,12.121729,8.301519,12.122545,10.921890,10.0,110.0,...,12.058087,Public Comprehensive EmOC,0.000000,0.000000,8555.357581,1.0,0.000117,0.000000,0.000038,0.004124
3,2,8.319272,12.072376,8.318758,12.071968,8.319786,12.072784,11.756603,1.0,8.0,...,12.107341,Public Comprehensive EmOC,0.059205,0.696052,4406.686022,1.0,0.000227,0.000013,0.000013,0.001466
4,2,8.319272,12.072376,8.318758,12.071968,8.319786,12.072784,11.756603,1.0,8.0,...,12.056152,Public Comprehensive EmOC,0.000000,0.000000,10659.897476,1.0,0.000094,0.000000,0.000013,0.001466


In [63]:
# Check the maximum value of the standardized Accessibility Index to ensure it is correctly normalized to 1.
max(origin_dest_acc.Accessibility_standard)

1.0

In [64]:
# Convert the final DataFrame to a GeoDataFrame and save it as a GeoPackage
gdf = gpd.GeoDataFrame(origin_dest_acc, geometry='geometry', crs="EPSG:4326")
gpkg_path = data_temp + 'accessibility-index.gpkg'
gdf.to_file(gpkg_path, layer="accessibility-index", driver="GPKG")

# 4. Grouping by grid ID to prepare the final output file
There is a need to update this part of the code

In [79]:
# Read the GeoPackage file (if starting from this section)
results_grid = gpd.read_file(data_temp + 'accessibility-index.gpkg')

# Select columns to keep and reorder them
results_grid = results_grid[['grid_id', 'origin_lon', 'origin_lat', 'origin_lon_min', 
                             'origin_lat_min', 'origin_lon_max', 'origin_lat_max', 'hcf_id', 
                             'facility_name', 'Local_Validation', 'duration_seconds', 'distance_km', 
                             'Accessibility_standard', 'geometry']]

### Setting values for Low medium and High categories

We started by defining equal value division, and modified the thesholds to a value that is more legible and easier to interpret. Every model should have their own thresholds based on the data distribution of the three categories. 

Note: For Kano, we excluded grid cells with index values below 0.000001 that indicated very low population and a small number of buildings.  

In [80]:
# Classify the accessibility into three categories based on the standardized Accessibility Index: 0 for High accessibility, 1 for Medium accessibility, and 2 for Low accessibility. The thresholds for classification are set at 0.005 and 0.02, which can be adjusted based on the distribution of the Accessibility Index in the dataset.
results_grid['result'] = 2  # Initialize all cells to 2 (Low accessibility)
results_grid.loc[results_grid['Accessibility_standard'] > 0.005, 'result'] = 1
results_grid.loc[results_grid['Accessibility_standard'] > 0.02, 'result'] = 0

In [81]:
results_grid.to_file(data_temp + 'accessibility-index-classified.gpkg', 
                     layer='accessibility-index-classified', driver='GPKG')

In [82]:
# The E2SFCA accessibility index is calculated based on the 3 closest healthcare 
# facilities for each grid cell. Since all 3 rows per grid cell share the same 
# accessibility value, we select the row with the minimum duration_seconds for each grid_id as representative to avoid duplication.
idx = results_grid.groupby('grid_id')['duration_seconds'].idxmin()
results_grid_dedup = results_grid.loc[idx].reset_index(drop=True)
print(len(results_grid_dedup))
results_grid_dedup.head()

print(results_grid_dedup['Local_Validation'].value_counts())

167239
Local_Validation
Private Comprehensive EmOC    120825
Public Comprehensive EmOC      36954
Private Basic EmOC              9203
Public Basic EmOC                257
Name: count, dtype: int64


In [83]:
category_counts = results_grid_dedup['result'].value_counts()
print(category_counts)

result
2    130081
1     24886
0     12272
Name: count, dtype: int64


### Setting values for focus areas

We defined the focus areas based on values for the different thresholds. We aim at participants helping us to confirm the selection of the city-specific thresholds.

In [70]:
# Identify focus areas for intervention based on the standardized Accessibility Index.
results_grid_dedup['focused'] = 0
# Focus areas between the Low category and the excluded cells due to low population or no buildings
results_grid_dedup.loc[(results_grid_dedup['Accessibility_standard'] > 0.000001) & (results_grid_dedup['Accessibility_standard'] < 0.0000015), 'focused'] = 1
# Focus areas between the Medium and High categories
results_grid_dedup.loc[(results_grid_dedup['Accessibility_standard'] > 0.003) & (results_grid_dedup['Accessibility_standard'] < 0.006), 'focused'] = 1
# Focus areas between the Low and Medium categories
results_grid_dedup.loc[(results_grid_dedup['Accessibility_standard'] > 0.019) & (results_grid_dedup['Accessibility_standard'] < 0.03), 'focused'] = 1

In [71]:
category_counts = results_grid_dedup['focused'].value_counts()
print(category_counts)

focused
0    147064
1     20175
Name: count, dtype: int64


In [72]:
# Rename columns for clarity and consistency
results_grid_dedup = results_grid_dedup.rename(columns={
    'origin_lon': 'longitude',
    'origin_lat': 'latitude',
    'origin_lon_min': 'lon_min',
    'origin_lat_min': 'lat_min',
    'origin_lon_max': 'lon_max',
    'origin_lat_max': 'lat_max',
    'Accessibility_standard': 'Accessibility_Index_Standard'
})

results_grid_dedup.head()

,grid_id,longitude,latitude,lon_min,lat_min,lon_max,lat_max,hcf_id,facility_name,Local_Validation,duration_seconds,distance_km,Accessibility_Index_Standard,geometry,result,focused
0,1,8.301005,12.122137,8.300491,12.121729,8.301519,12.122545,14,Dawakin Tofa General Hospital,Public Comprehensive EmOC,374.30,5.77,0.004124,"POLYGON ((8.30051 12.12254, 8.30152 12.12254, ...",2,1
1,2,8.319272,12.072376,8.318758,12.071968,8.319786,12.072784,14,Dawakin Tofa General Hospital,Public Comprehensive EmOC,470.08,5.20,0.001466,"POLYGON ((8.31877 12.07278, 8.31979 12.07278, ...",2,0
2,3,8.330126,12.110716,8.329612,12.110308,8.330640,12.111124,14,Dawakin Tofa General Hospital,Public Comprehensive EmOC,104.01,0.48,0.021559,"POLYGON ((8.32963 12.11112, 8.33064 12.11112, ...",0,1
3,4,8.330079,12.108269,8.329565,12.107861,8.330593,12.108676,14,Dawakin Tofa General Hospital,Public Comprehensive EmOC,40.99,0.22,0.024239,"POLYGON ((8.32958 12.10868, 8.33059 12.10868, ...",0,1
4,5,8.332575,12.027513,8.332061,12.027105,8.333088,12.027921,14,Dawakin Tofa General Hospital,Public Comprehensive EmOC,1518.11,12.65,0.000000,"POLYGON ((8.33208 12.02792, 8.33309 12.02792, ...",2,0


In [74]:
# Dataset 2: 
dataset_2 = results_grid_dedup.copy()
dataset_2 = gpd.GeoDataFrame(
    results_grid_dedup,
    geometry='geometry',
    crs='EPSG:4326'
)

dataset_2 = dataset_2[['grid_id', 'longitude', 'latitude', 'lon_min', 'lon_max', 'lat_min', 'lat_max', 'Accessibility_Index_Standard', 'result', 
                         'focused', 'geometry']]
output_gpkg_path = model_outputs + 'deprivation-classification.gpkg'
dataset_2.to_file(output_gpkg_path, layer='deprivation-classification', driver='GPKG')
print(f"\n✓ Saved dataset_2 with {len(dataset_2)} records to {output_gpkg_path}")
dataset_2.head()


✓ Saved dataset_2 with 167239 records to ../Kano/Data/outputs/deprivation-classification.gpkg


,grid_id,longitude,latitude,lon_min,lon_max,lat_min,lat_max,Accessibility_Index_Standard,result,focused,geometry
0,1,8.301005,12.122137,8.300491,8.301519,12.121729,12.122545,0.004124,2,1,"POLYGON ((8.30051 12.12254, 8.30152 12.12254, ..."
1,2,8.319272,12.072376,8.318758,8.319786,12.071968,12.072784,0.001466,2,0,"POLYGON ((8.31877 12.07278, 8.31979 12.07278, ..."
2,3,8.330126,12.110716,8.329612,8.330640,12.110308,12.111124,0.021559,0,1,"POLYGON ((8.32963 12.11112, 8.33064 12.11112, ..."
3,4,8.330079,12.108269,8.329565,8.330593,12.107861,12.108676,0.024239,0,1,"POLYGON ((8.32958 12.10868, 8.33059 12.10868, ..."
4,5,8.332575,12.027513,8.332061,8.333088,12.027105,12.027921,0.000000,2,0,"POLYGON ((8.33208 12.02792, 8.33309 12.02792, ..."


In [ ]:
dataset_2 = gpd.read_file(model_outputs + 'deprivation-classification.gpkg')
dataset_2.to_csv(model_outputs + 'deprivation-classification.csv', index=False)